In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
from sklearn.model_selection import learning_curve

In [ ]:
X = data.drop(columns=["stroke_risk_percentage"])
y = data["stroke_risk_percentage"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

In [ ]:
# Models
lr = LinearRegression()
dt = DecisionTreeRegressor(random_state=42)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
knn = KNeighborsRegressor(n_neighbors=5)

# Train
lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
rf.fit(X_train, y_train)
knn.fit(X_train, y_train)

In [ ]:
models = {
    "Linear Regression": (lr, X_test),
    "Decision Tree": (dt, X_test),
    "Random Forest": (rf, X_test),
    "KNN": (knn, X_test)
}

for name, (model, X_data) in models.items():
    y_pred = model.predict(X_data)

    print(f"{name}")
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("R2:", r2_score(y_test, y_pred))


In [ ]:
# Predicted vs Actual
results = {}    # Store predictions

models = {
    "Linear Regression": (lr, X_test),
    "Decision Tree": (dt, X_test),
    "Random Forest": (rf, X_test),
    "KNN": (knn, X_test)
}

# Evaluation
for name, (model, X_data) in models.items():
    y_pred = model.predict(X_data)
    results[name] = {'y_pred': y_pred}

# Plot Predicted vs Actual
plt.figure(figsize=(8,6))
for name, r in results.items():
    plt.scatter(y_test, r['y_pred'], alpha=0.5, label=name)

# Perfect prediction line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'k--',
    lw=2,
    label="Perfect Prediction")

plt.xlabel('Actual stroke_risk_percentage')
plt.ylabel('Predicted stroke_risk_percentage')
plt.title('Predicted vs Actual')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Residual plots for each model
fig, axes = plt.subplots(1, len(results), figsize=(5*len(results),4))
if len(results) == 1:
    axes = [axes]
for ax, (name, r) in zip(axes, results.items()):
    residuals = y_test - r['y_pred']
    sns.histplot(residuals, kde=True, ax=ax)
    ax.set_title(name + " Residuals")
    ax.axvline(0, color='k', linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# Stacking
estimators = [
    ('lr', LinearRegression()),
    ('dt', DecisionTreeRegressor(random_state=42)),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42,min_samples_split=2, min_samples_leaf=1 , max_depth=20)),
    ('knn', KNeighborsRegressor(n_neighbors=5))
]

stack_model = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression()
)

stack_model.fit(X_train, y_train)

y_pred_stack = stack_model.predict(X_test)

print("Stacking")
print("MAE:", mean_absolute_error(y_test, y_pred_stack))
print("MSE:", mean_squared_error(y_test, y_pred_stack))
print("R2:", r2_score(y_test, y_pred_stack))

In [ ]:
# Define estimators and parameter grids
param_grids = {
    "RandomForest": {
        'estimator': RandomForestRegressor(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2]
        }
    },
    "DecisionTree": {
        'estimator': DecisionTreeRegressor(random_state=42),
        'params': {
            'max_depth': [None, 5, 10, 20],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    "KNN": {
        'estimator': Pipeline([('scaler', StandardScaler()), ('reg', KNeighborsRegressor())]),
        'params': {
            'reg__n_neighbors': [3, 5, 7],
            'reg__weights': ['uniform', 'distance']
        }
    }
}

# Choose scoring metric for CV (use neg_mean_squared_error or neg_mean_absolute_error)
scoring = 'neg_mean_squared_error'
cv = 5
n_jobs = -1

best_models = {}

for name, cfg in param_grids.items():
    print(f"Running GridSearchCV for {name} ...")
    gs = GridSearchCV(cfg['estimator'], cfg['params'], scoring=scoring, cv=cv, n_jobs=n_jobs, verbose=1)
    gs.fit(X_train, y_train)
    best_models[name] = gs.best_estimator_
    print(f"Best {name} params: {gs.best_params_}")
    print(f"Best CV score ({scoring}): {gs.best_score_:.4f}\n")

# Evaluate best models on test set
for name, model in best_models.items():
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name} test -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")


In [ ]:
rf = best_models["RandomForest"]

importances = rf.feature_importances_
feature_names = X_train.columns

indices = np.argsort(importances)[::-1]

plt.figure()
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), feature_names[indices], rotation=90)
plt.title("Feature Importance (Random Forest)")
plt.tight_layout()
plt.show()

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best_models["RandomForest"], X_train, y_train, cv=5, scoring='neg_mean_squared_error'
)

train_scores = -train_scores.mean(axis=1)
val_scores = -val_scores.mean(axis=1)

plt.figure()
plt.plot(train_sizes, train_scores, label="Train Error")
plt.plot(train_sizes, val_scores, label="Validation Error")
plt.legend()
plt.title("Learning Curve")
plt.show()

**[2] XAI Techniques**

1. LIME

In [ ]:
!pip install lime

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_exp = LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X.columns.tolist(),
    mode='regression'
)
num_instances = 5
for i in range(num_instances):
    print(f"\n Explanation for instance {i}\n")

    exp = lime_exp.explain_instance(
        X_test.iloc[i].values,
        stack_model.predict
    )
    exp.show_in_notebook(show_table=True)

2. Shap

In [ ]:
background = shap.sample(X_train, 100).values
X_test_sample = X_test.iloc[:50].values


In [ ]:
def model_predict(x):
    return stack_model.predict(x)

In [ ]:
background = shap.sample(X_train, 100).values
X_test_sample = X_test.iloc[:50].values

def model_predict(x):
    return stack_model.predict(x)

# Create explainer
explainer = shap.KernelExplainer(model_predict, background)
# Compute SHAP values
shap_values = explainer.shap_values(X_test_sample)

# Plot
shap.summary_plot(
    shap_values,
    X_test_sample,
    feature_names=X_train.columns
)